In [81]:
import pandas as pd

df = pd.read_csv("../data/processed/cleeaned_fraud_train.csv")
df

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,trans_num,unix_time,merch_lat,merch_long,is_fraud,date_of_birth,week_day,hour,age,age_group
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0,1988-03-09,Tuesday,0,31,25-34
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,...,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0,1978-06-21,Tuesday,0,41,35-49
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,...,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0,1962-01-19,Tuesday,0,57,50-64
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,...,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0,1967-01-12,Tuesday,0,52,50-64
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,...,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0,1986-03-28,Tuesday,0,33,25-34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296670,2020-06-21 12:12:08,30263540414123,fraud_Reichel Inc,entertainment,15.56,Erik,Patterson,M,162 Jessica Row Apt. 072,Hatch,...,440b587732da4dc1a6395aba5fb41669,1371816728,36.841266,-111.690765,0,1961-11-24,Sunday,12,59,50-64
1296671,2020-06-21 12:12:19,6011149206456997,fraud_Abernathy and Sons,food_dining,51.70,Jeffrey,White,M,8617 Holmes Terrace Suite 651,Tuscarora,...,278000d2e0d2277d1de2f890067dcc0a,1371816739,38.906881,-78.246528,0,1979-12-11,Sunday,12,41,35-49
1296672,2020-06-21 12:12:32,3514865930894695,fraud_Stiedemann Ltd,food_dining,105.93,Christopher,Castaneda,M,1632 Cohen Drive Suite 639,High Rolls Mountain Park,...,483f52fe67fabef353d552c1e662974c,1371816752,33.619513,-105.130529,0,1967-08-30,Sunday,12,53,50-64
1296673,2020-06-21 12:13:36,2720012583106919,"fraud_Reinger, Weissnat and Strosin",food_dining,74.90,Joseph,Murray,M,42933 Ryan Underpass,Manderson,...,d667cdcbadaaed3da3f4020e83591c83,1371816816,42.788940,-103.241160,0,1980-08-18,Sunday,12,40,35-49


# Fraud KPIs

In [82]:
def get_total_transactions(df:pd.DataFrame):
    return len(df)

def get_total_fraud_transactions(df:pd.DataFrame):
    df_fraud = df[df["is_fraud"] == 1]
    return len(df_fraud)

def get_fraud_rate(df):
    total_fraud = get_total_fraud_transactions(df)
    total = get_total_transactions(df)
    return total_fraud/total * 100

def get_total_sum(df:pd.DataFrame):
    return df['amt'].sum()

def get_fraud_loss_sum(df:pd.DataFrame):
    df_fraud = df[df["is_fraud"] == 1]
    return df_fraud["amt"].sum()

def get_fraud_loss_rate(df:pd.DataFrame):
    total_sum = get_total_sum(df)
    fraud_sum = get_fraud_loss_sum(df)
    return fraud_sum/total_sum * 100



In [83]:
print(get_total_transactions(df), get_total_fraud_transactions(df))

1296675 7506


In [84]:
print(f"fraud rate is {get_fraud_rate(df):.2}%, total loss rate from fraud is {get_fraud_loss_rate(df):.2}%")

fraud rate is 0.58%, total loss rate from fraud is 4.4%


# Time Analysis

In [85]:
def get_fraud_rate_by_hour(df:pd.DataFrame):
    df_fraud = df[df["is_fraud"] == 1]
    fraud_hour_df = df_fraud.groupby(['hour']).count()["first"]

    hour_df = df.groupby(['hour']).count()["first"]

    fraud_rate_by_hour = fraud_hour_df / hour_df * 100
    
    return fraud_rate_by_hour

def get_fraud_rate_by_day(df:pd.DataFrame):
    df_fraud = df[df["is_fraud"] == 1]
    fraud_week_day_df = df_fraud.groupby(['week_day']).count()["first"]

    hour_df = df.groupby(['week_day']).count()["first"]

    fraud_rate_by_week_day = fraud_week_day_df / hour_df * 100
    
    return fraud_rate_by_week_day

# 3. Category & Merchant Risk


In [86]:
def get_category_fraud_risk_grouped(df:pd.DataFrame):
    category_frauds = df.groupby(["category","is_fraud"]).count()[["first"]]
    return category_frauds.reset_index()

def get_category_fraud_risk_rate(df:pd.DataFrame):
    df_grouped = get_category_fraud_risk_grouped(df)
    df_frauds = df_grouped[df_grouped["is_fraud"] == 1][["category", "first"]].reset_index()
    df_totals = df_grouped[df_grouped["is_fraud"] == 0][["category", "first"]].reset_index()
    df_totals["risk"] = df_frauds["first"] / df_totals["first"] * 100
    df_totals.drop(columns=['first', "index"],inplace=True) 
    return df_totals

get_category_fraud_risk_rate(df)
    

,category,risk
0,entertainment,0.248451
1,food_dining,0.165371
2,gas_transport,0.471608
3,grocery_net,0.295688
4,grocery_pos,1.429919
5,health_fitness,0.155109
6,home,0.161084
7,kids_pets,0.211887
8,misc_net,1.467004
9,misc_pos,0.314842


# 4. Customer Segments

In [88]:
# def get_fraud_by_age_group(df:pd.DataFrame):
    
df.groupby(["age_group", 'is_fraud']).count()[["first"]].reset_index()

,age_group,is_fraud,first
0,0-17,0,13370
1,0-17,1,60
2,18-24,0,96831
3,18-24,1,660
4,25-34,0,276991
5,25-34,1,1370
6,35-49,0,435448
7,35-49,1,1955
8,50-64,0,265721
9,50-64,1,1957
